# Paper 4 — 05 · Explaining Paper 3 (H1d)

Map Paper 3's `selected_blocks.json` (the refusal-direction top-k it trained on) onto this paper's detection/execution band map. H1d predicts they fall in the **execution** band. Then one confirmatory **detection-band-targeted** DPO run per anchor, reusing Paper 3's `03_train_rd_dpo` + `04_eval_safety` wholesale, comparing gap closure (EXPERIMENT_DESIGN §7).

**Output:** `results/<short>/paper3_crossref.json`.

In [ ]:
%%capture
# Pinned to requirements.txt. Wheel-only on A100 / CUDA 12; restart rarely needed.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    'transformer-lens>=2.9' \
    'sae-lens>=4.0' \
    python-dotenv requests huggingface_hub ipywidgets pyyaml matplotlib seaborn -q


In [ ]:
import os, json, gc, sys, hashlib
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Secrets (Colab) -> env, so the Paper 2 judge + gated HF models work
#     end-to-end with no manual steps. Set these in Colab -> Secrets first. ---
try:
    from google.colab import userdata
    for _k in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            print(f"[secrets] {_k} not set in Colab Secrets — add it if a cell needs it.")
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Reuse Paper 2 judge harness + Paper 3 helpers; Paper 4 src/ ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(DRIVE_ROOT / "src"))        # paths, capture, probes, patching, sae_utils, contrastive, behavioral

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## Configuration

In [ ]:
# --- Anchor selection. Re-run the notebook once per anchor. ---
# SAE anchor (H1e available):  google/gemma-2-2b-it
# Cross-arch anchors:          Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct
ANCHOR = "google/gemma-2-2b-it"

from paths import short_of, family_of
short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}\nfamily : {family}\nshort  : {short}")


In [ ]:
assert short in ('qwen2.5-3b', 'llama-3.2-3b'), 'H1d uses the two shared Paper-3 anchors.'

## 1. Band membership of Paper 3's refusal-direction blocks

H1d predicts Paper 3's `selected_blocks` (the refusal-direction top-k it trained
on) fall in the **execution** band, explaining why RD-DPO couldn't repair an
upstream detection deficit.

In [ ]:
p3 = json.loads((PAPER3_ROOT / 'data' / 'probes' / short / 'selected_blocks.json').read_text())
bands = json.loads((RESULTS_DIR / short / 'bands.json').read_text())
sel = p3.get('4') or p3.get(4)   # k=4, matched to Paper 3
in_exec = [b for b in sel if b in bands['execution']]
in_det  = [b for b in sel if b in bands['detection']]
print(f'Paper 3 k=4 blocks: {sel}')
print(f'  in execution band {bands["execution"]}: {in_exec}')
print(f'  in detection band {bands["detection"]}: {in_det}')
h1d_supported = len(in_exec) > len(in_det)
print('H1d (blocks are execution-band):', h1d_supported)

## 2. Emit detection-band target blocks for the confirmatory DPO run

Top-k by **detection**-probe accuracy within the detection band (k matched to
Paper 3). Feed these to Paper 3's `03_train_rd_dpo` as a `target_blocks` override.

In [ ]:
lp = json.loads((RESULTS_DIR / short / 'linear_probes.json').read_text())
det_layers = sorted(bands['detection'],
                    key=lambda L: lp['per_layer'][L]['det_acc_en'], reverse=True)[:4]
det_layers = sorted(det_layers)
target = {'4': det_layers}
p3_override = PAPER3_ROOT / 'data' / 'probes' / short / 'selected_blocks_detection.json'
p3_override.write_text(json.dumps(target, indent=2))
print('detection-band target blocks (k=4):', det_layers)
print('wrote override for Paper 3 ->', p3_override)

## 3. Confirmatory run (manual, reuses Paper 3 wholesale)

In the Paper 3 repo, run `experiments/03_train_rd_dpo.ipynb` with
`target_blocks = selected_blocks_detection.json` (one seed pilot, then 3 seeds
if promising), then `04_eval_safety.ipynb` on the RoSafetyBench holdout. Both
outcomes are publishable: detection-band > execution-band confirms H1d; both
failing shows the deficit isn't LoRA-repairable at this budget (localization
stands independently).

## 4. Compare gap closure (auto-loads Paper 3 eval results if present)

In [ ]:
def _load_safety(cond):
    f = PAPER3_ROOT / 'results' / f'{short}__{cond}__seed17__safety.json'
    return json.loads(f.read_text()) if f.exists() else None
exec_dpo = _load_safety('rd-dpo-k4-bal-e6-x4')      # Paper 3's execution-band selection
det_dpo  = _load_safety('rd-dpo-k4-detection')       # the confirmatory detection-band run
result = {'anchor_model': ANCHOR, 'short': short, 'analysis': 'paper3_crossref',
          'paper3_selected_k4': sel, 'bands': bands,
          'selected_in_execution': in_exec, 'selected_in_detection': in_det,
          'h1d_blocks_are_execution': h1d_supported,
          'detection_band_target': det_layers,
          'exec_dpo_present': exec_dpo is not None, 'det_dpo_present': det_dpo is not None}
(RESULTS_DIR / short / 'paper3_crossref.json').write_text(json.dumps(result, indent=2))
print(json.dumps({k: result[k] for k in ['h1d_blocks_are_execution','detection_band_target','det_dpo_present']}, indent=2))